# PZ Companion — llama.cpp Notebook Test

这个 notebook 用于实验性测试 4 个 fixture：

- `player_command_follow.json`
- `player_command_pickup.json`
- `event_batch_noise.json`
- `event_batch_threat.json`

前提：你的本机已经启动 `llama-server`，默认地址为：

`http://127.0.0.1:8080`


In [2]:
import json
import time
import urllib.request
import urllib.error
from pathlib import Path

BASE_URL = "http://127.0.0.1:8080"
CHAT_URL = f"{BASE_URL}/v1/chat/completions"

print("Using:", CHAT_URL)


Using: http://127.0.0.1:8080/v1/chat/completions


## 1. 检查 llama.cpp server

先运行下面这个 cell。  
如果成功，应看到 server health response；如果失败，先确认 `llama-server.exe` 是否已经启动。


In [3]:
try:
    with urllib.request.urlopen(f"{BASE_URL}/health", timeout=3) as resp:
        print(resp.read().decode("utf-8"))
except Exception as e:
    print("Server not reachable:", repr(e))


{"status":"ok"}


## 2. 定义 action schema 和 prompt

In [4]:
ACTION_SET = [
    {"name": "continue_follow", "desc": "Nothing changes, keep following the player"},
    {"name": "move_to", "desc": "Move to the given coordinates (target_x, target_y)"},
    {"name": "investigate", "desc": "Go check out something that caught attention; needs target_x/target_y"},
    {"name": "pick_up_item", "desc": "Pick up an item; needs target_id identifying the item"},
    {"name": "retreat", "desc": "Fall back to the player's side; use when there's a sensed threat but not yet active combat"},
    {"name": "wait", "desc": "Stay put and take no action"},
]

ACTION_NAMES = [a["name"] for a in ACTION_SET]

ACTION_SCHEMA = {
    "type": "object",
    "properties": {
        "action": {"type": "string", "enum": ACTION_NAMES},
        "target_x": {"type": ["number", "null"]},
        "target_y": {"type": ["number", "null"]},
        "target_id": {"type": ["string", "null"]},
        "reason": {"type": "string"},
    },
    "required": ["action", "target_x", "target_y", "target_id", "reason"],
}


In [5]:
def build_system_prompt():
    action_lines = "\n".join(
        f"- {a['name']}: {a['desc']}" for a in ACTION_SET
    )

    return f"""You are the decision-making module for a human survivor companion NPC in Project Zomboid.

            Your job: given the current situation, choose exactly one action from the
            fixed set below that best fits. You must not output anything outside this
            action set.

            Available actions:
            {action_lines}

            Rules:
            1. Only choose one action name from the list above.
            2. Fields you don't need must be null.
            3. If the player's intent is unclear, default to continue_follow.
            4. The reason field must be one short sentence.

            Examples:
            Input: player says "follow me" -> Output: {{"action":"continue_follow","target_x":null,"target_y":null,"target_id":null,"reason":"player explicitly asked to keep following"}}
            Input: player says "go grab that axe", axe_01 is nearby -> Output: {{"action":"pick_up_item","target_x":null,"target_y":null,"target_id":"axe_01","reason":"player specified a clear pickup target"}}
            Input: event report "strange noise heard nearby, at (120,340)" -> Output: {{"action":"investigate","target_x":120,"target_y":340,"target_id":null,"reason":"a non-urgent but worth-checking event occurred"}}
            Input: event report "threat sensed but not yet under attack" -> Output: {{"action":"retreat","target_x":null,"target_y":null,"target_id":null,"reason":"potential threat present"}}
            """


def build_user_prompt(request):
    trigger_type = request.get("trigger_type")
    state = request.get("state", {})

    state_summary = (
        f"Player health: {state.get('health', 'unknown')}\n"
        f"Nearby zombies: {len(state.get('nearby_zombies', []))}\n"
        f"Nearby items: {state.get('nearby_items', [])}\n"
    )

    if trigger_type == "player_command":
        return (
            "[Triggered by player command]\n"
            f'Player said: "{request.get("player_command", "")}"\n\n'
            f"Current state:\n{state_summary}\n"
            "Choose one action."
        )

    if trigger_type == "event_batch":
        events = request.get("events", [])
        event_lines = "\n".join(
            f"- {e.get('type')}: {e.get('detail', '')}" for e in events
        )
        return (
            "[Triggered by an event report, no player command]\n"
            f"The following happened recently:\n{event_lines}\n\n"
            f"Current state:\n{state_summary}\n"
            "Choose one action."
        )

    raise ValueError(f"Unknown trigger_type: {trigger_type}")


## 3. llama.cpp API 调用函数

In [6]:
def call_llama_cpp(system_prompt, user_prompt, timeout=20):
    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "response_format": {
            "type": "json_schema",
            "schema": ACTION_SCHEMA,
        },
        "temperature": 0,
        "stream": False,
    }

    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        CHAT_URL,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    start = time.time()
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        body = json.loads(resp.read().decode("utf-8"))

    elapsed = time.time() - start
    raw_content = body["choices"][0]["message"]["content"]
    intent = json.loads(raw_content)

    return {
        "intent": intent,
        "elapsed_seconds": round(elapsed, 3),
    }


## 4. 加载 4 个 fixtures

如果 notebook 和 JSON 文件在同一个文件夹，下面代码可直接运行。  
如果你的 JSON 在 `fixtures/` 文件夹，把 `FIXTURE_DIR = Path(".")` 改成 `Path("fixtures")`。


In [7]:
FIXTURE_DIR = Path(".")

fixture_files = {
    "follow": "player_command_follow.json",
    "pickup": "player_command_pickup.json",
    "noise": "event_batch_noise.json",
    "threat": "event_batch_threat.json",
}

fixtures = {}

for name, filename in fixture_files.items():
    path = FIXTURE_DIR / filename
    with open(path, "r", encoding="utf-8") as f:
        fixtures[name] = json.load(f)

fixtures


{'follow': {'trigger_type': 'player_command',
  'player_command': 'follow me',
  'state': {'health': 90, 'nearby_zombies': [], 'nearby_items': []}},
 'pickup': {'trigger_type': 'player_command',
  'player_command': 'go grab that axe',
  'state': {'health': 90,
   'nearby_zombies': [],
   'nearby_items': [{'id': 'axe_01', 'name': 'axe', 'x': 118, 'y': 342}]}},
 'noise': {'trigger_type': 'event_batch',
  'events': [{'type': 'noise_heard',
    'detail': 'a strange noise heard nearby, at (120,340)'}],
  'state': {'health': 85, 'nearby_zombies': [], 'nearby_items': []}},
 'threat': {'trigger_type': 'event_batch',
  'events': [{'type': 'zombie_spotted',
    'detail': 'two zombies approaching from 10 tiles away, have not noticed the player yet'}],
  'state': {'health': 85,
   'nearby_zombies': [{'x': 130, 'y': 350}, {'x': 135, 'y': 355}],
   'nearby_items': []}}}

## 5. 先只看模型实际收到的 prompt

In [8]:
system_prompt = build_system_prompt()

for name, request in fixtures.items():
    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)
    print(build_user_prompt(request))



FOLLOW
[Triggered by player command]
Player said: "follow me"

Current state:
Player health: 90
Nearby zombies: 0
Nearby items: []

Choose one action.

PICKUP
[Triggered by player command]
Player said: "go grab that axe"

Current state:
Player health: 90
Nearby zombies: 0
Nearby items: [{'id': 'axe_01', 'name': 'axe', 'x': 118, 'y': 342}]

Choose one action.

NOISE
[Triggered by an event report, no player command]
The following happened recently:
- noise_heard: a strange noise heard nearby, at (120,340)

Current state:
Player health: 85
Nearby zombies: 0
Nearby items: []

Choose one action.

THREAT
[Triggered by an event report, no player command]
The following happened recently:
- zombie_spotted: two zombies approaching from 10 tiles away, have not noticed the player yet

Current state:
Player health: 85
Nearby zombies: 2
Nearby items: []

Choose one action.


## 6. 一次跑完 4 个场景

In [14]:
results = {}
summary=[]

for name, request in fixtures.items():
    user_prompt = build_user_prompt(request)

    try:
        result = call_llama_cpp(system_prompt, user_prompt)
        summary.append(results)
        results[name] = result

        print("\n" + "=" * 80)
        print(name.upper())
        print("Elapsed:", result["elapsed_seconds"], "seconds")
        print(json.dumps(result["intent"], ensure_ascii=False, indent=2))

    except Exception as e:
        results[name] = {"error": repr(e)}
        print("\n" + "=" * 80)
        print(name.upper(), "FAILED")
        print(repr(e))



FOLLOW
Elapsed: 1.091 seconds
{
  "action": "continue_follow",
  "target_x": null,
  "target_y": null,
  "target_id": null,
  "reason": "player explicitly asked to keep following"
}

PICKUP
Elapsed: 1.397 seconds
{
  "action": "pick_up_item",
  "target_x": null,
  "target_y": null,
  "target_id": "axe_01",
  "reason": "player explicitly asked to grab the axe nearby"
}

NOISE
Elapsed: 1.566 seconds
{
  "action": "investigate",
  "target_x": 120,
  "target_y": 340,
  "target_id": null,
  "reason": "a strange noise heard nearby, at (120,340)"
}

THREAT
Elapsed: 1.291 seconds
{
  "action": "investigate",
  "target_x": null,
  "target_y": null,
  "target_id": null,
  "reason": "zombie_spotted event detected with two approaching zombies"
}


## 7. 自动检查是否符合当前预期

按照你目前 prompt 中的行为定义：

- follow → `continue_follow`
- pickup → `pick_up_item`, `target_id == axe_01`
- noise → `investigate`, 坐标 `(120, 340)`
- threat → `retreat`


In [11]:
expected = {
    "follow": {"action": "continue_follow"},
    "pickup": {"action": "pick_up_item", "target_id": "axe_01"},
    "noise": {"action": "investigate", "target_x": 120, "target_y": 340},
    "threat": {"action": "retreat"},
}

for name, exp in expected.items():
    result = results.get(name, {})
    if "error" in result:
        print(f"{name}: ❌ API error -> {result['error']}")
        continue

    intent = result["intent"]
    ok = all(intent.get(k) == v for k, v in exp.items())

    print(
        f"{name}: {'✅ PASS' if ok else '❌ FAIL'} "
        f"| expected={exp} | actual={intent}"
    )


follow: ✅ PASS | expected={'action': 'continue_follow'} | actual={'action': 'continue_follow', 'target_x': None, 'target_y': None, 'target_id': None, 'reason': 'player explicitly asked to keep following'}
pickup: ✅ PASS | expected={'action': 'pick_up_item', 'target_id': 'axe_01'} | actual={'action': 'pick_up_item', 'target_x': None, 'target_y': None, 'target_id': 'axe_01', 'reason': 'player explicitly asked to grab the axe nearby'}
noise: ✅ PASS | expected={'action': 'investigate', 'target_x': 120, 'target_y': 340} | actual={'action': 'investigate', 'target_x': 120, 'target_y': 340, 'target_id': None, 'reason': 'a strange noise heard nearby, at (120,340)'}
threat: ❌ FAIL | expected={'action': 'retreat'} | actual={'action': 'investigate', 'target_x': None, 'target_y': None, 'target_id': None, 'reason': 'zombie_spotted event detected with two approaching zombies'}


## 8. 延迟汇总

In [15]:
latencies = {
    name: result.get("elapsed_seconds")
    for name, result in results.items()
    if "elapsed_seconds" in result
}

latencies


{'follow': 1.091, 'pickup': 1.397, 'noise': 1.566, 'threat': 1.291}